In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("Downloads/cleaned_dataset_retail-project.csv")

In [4]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,100000,P1003,Laptop Stand,4,2023-12-16,35.00,10265,UNITED KINGDOM,140.00
1,100000,P1005,HD Webcam,4,2023-07-08,65.00,10937,SINGAPORE,260.00
2,100000,P1001,Wireless Mouse,2,2024-08-01,15.99,11956,GERMANY,31.98
3,100001,P1004,Noise Cancelling Headphones,5,2023-10-31,120.00,11547,UAE,600.00
4,100001,P1008,External SSD 1TB,5,2024-05-21,140.00,11980,INDIA,700.00


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    1000 non-null   int64  
 1   StockCode    1000 non-null   object 
 2   Description  1000 non-null   object 
 3   Quantity     1000 non-null   int64  
 4   InvoiceDate  1000 non-null   object 
 5   UnitPrice    1000 non-null   float64
 6   CustomerID   1000 non-null   int64  
 7   Country      1000 non-null   object 
 8   TotalPrice   1000 non-null   float64
dtypes: float64(2), int64(3), object(4)
memory usage: 70.4+ KB


In [6]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    1000 non-null   int64         
 1   StockCode    1000 non-null   object        
 2   Description  1000 non-null   object        
 3   Quantity     1000 non-null   int64         
 4   InvoiceDate  1000 non-null   datetime64[ns]
 5   UnitPrice    1000 non-null   float64       
 6   CustomerID   1000 non-null   int64         
 7   Country      1000 non-null   object        
 8   TotalPrice   1000 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 70.4+ KB


In [16]:
# RFM Analysis

In [8]:
reference_date = df["InvoiceDate"].max()

In [39]:
print(reference_date)

2024-12-30 00:00:00


In [9]:
recency = df.groupby("CustomerID")["InvoiceDate"].max()
recency = (reference_date-recency).dt.days

In [44]:
print(recency)

CustomerID
10002    85 days
10004   321 days
10006   128 days
10008   550 days
10009   273 days
          ...   
11991   540 days
11992   682 days
11993   468 days
11994    60 days
11997   275 days
Name: InvoiceDate, Length: 794, dtype: timedelta64[ns]


In [10]:
frequency = df.groupby("CustomerID")["InvoiceNo"].count()

In [48]:
print(frequency)

CustomerID
10002    3
10004    3
10006    1
10008    1
10009    1
        ..
11991    1
11992    1
11993    1
11994    1
11997    2
Name: InvoiceNo, Length: 794, dtype: int64


In [11]:
monetary = df.groupby("CustomerID")["TotalPrice"].sum()

In [51]:
print(monetary)

CustomerID
10002    552.00
10004    602.00
10006    130.00
10008    220.00
10009     70.00
          ...  
11991    127.50
11992     90.00
11993     90.00
11994    120.00
11997    138.96
Name: TotalPrice, Length: 794, dtype: float64


In [12]:
rfm = pd.concat ([recency, frequency,monetary],axis=1)


In [13]:
rfm.columns= ["recency","frequency","monetary"]
rfm.columns

Index(['recency', 'frequency', 'monetary'], dtype='object')

In [14]:
print(rfm)

            recency  frequency  monetary
CustomerID                              
10002            85          3    552.00
10004           321          3    602.00
10006           128          1    130.00
10008           550          1    220.00
10009           273          1     70.00
...             ...        ...       ...
11991           540          1    127.50
11992           682          1     90.00
11993           468          1     90.00
11994            60          1    120.00
11997           275          2    138.96

[794 rows x 3 columns]


In [19]:
rfm.head()

,recency,frequency,monetary
CustomerID,,,
10002,85,3,552.0
10004,321,3,602.0
10006,128,1,130.0
10008,550,1,220.0
10009,273,1,70.0


In [72]:
  # RFM Scoring

In [15]:
rfm["R_score"] = pd.qcut(rfm["recency"],5,
                         labels =[5,4,3,2,1])

In [16]:
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method= "first"),5,
                         labels = [1,2,3,4,5])

In [17]:
rfm["M_score"] = pd.qcut(rfm["monetary"],5,
                         labels = [1,2,3,4,5])

In [18]:
rfm["RFM score"] = (rfm["R_score"].astype(str)+
                    rfm["F_score"].astype(str)+
                    rfm["M_score"].astype(str))

In [19]:
rfm.head()

,recency,frequency,monetary,R_score,F_score,M_score,RFM score
CustomerID,,,,,,,
10002,85,3,552.0,5,5,5,555
10004,321,3,602.0,3,5,5,355
10006,128,1,130.0,4,1,3,413
10008,550,1,220.0,1,1,4,114
10009,273,1,70.0,3,1,2,312


In [20]:
 rfm["RFM score"].value_counts()

RFM score
555    35
131    17
455    16
355    15
454    13
       ..
152     1
153     1
135     1
351     1
451     1
Name: count, Length: 122, dtype: int64

In [ ]:
# customer segmentation

In [21]:
def segment_customer(row):
    r = row['R_score']
    f = row['F_score']
    m = row['M_score']

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    
    elif r >= 3 and f >= 4:
        return "Loyal Customers"
    
    elif r >= 4 and f >= 2 and m >= 2:
        return "Potential Loyalist"
    
    elif r >= 4 and f <= 2:
        return "Recent Users"
    
    elif r == 3 and f <= 2:
        return "Promising"
    
    elif r == 3 and f >= 3:
        return "Needs Attention"
    
    elif r == 2 and f <= 3:
        return "About To Sleep"
    
    elif r == 1 and f <= 2:
        return "Lost"
    
    elif r <= 2 and f >= 3:
        return "Hibernating"
    
    else:
        return "Price Sensitive"

In [22]:
rfm["Segment"] = rfm.apply(segment_customer, axis=1)

In [23]:
rfm["Segment"].value_counts()

Segment
Loyal Customers       135
Hibernating           133
About To Sleep        103
Champions              91
Lost                   82
Potential Loyalist     75
Recent Users           70
Promising              61
Needs Attention        32
Price Sensitive        12
Name: count, dtype: int64

In [30]:
rfm.head()

,recency,frequency,monetary,R_score,F_score,M_score,RFM score,Segment
CustomerID,,,,,,,,
10002,85,3,552.0,5,5,5,555,Champions
10004,321,3,602.0,3,5,5,355,Loyal Customers
10006,128,1,130.0,4,1,3,413,Recent Users
10008,550,1,220.0,1,1,4,114,Lost
10009,273,1,70.0,3,1,2,312,Promising


In [24]:
final_data = df.merge(rfm, on="CustomerID", how="left")

In [25]:
final_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,recency,frequency,monetary,R_score,F_score,M_score,RFM score,Segment
0,100000,P1003,Laptop Stand,4,2023-12-16,35.00,10265,UNITED KINGDOM,140.00,380,1,140.00,3,1,3,313,Promising
1,100000,P1005,HD Webcam,4,2023-07-08,65.00,10937,SINGAPORE,260.00,541,1,260.00,2,2,4,224,About To Sleep
2,100000,P1001,Wireless Mouse,2,2024-08-01,15.99,11956,GERMANY,31.98,151,1,31.98,4,4,1,441,Loyal Customers
3,100001,P1004,Noise Cancelling Headphones,5,2023-10-31,120.00,11547,UAE,600.00,426,1,600.00,2,4,5,245,Hibernating
4,100001,P1008,External SSD 1TB,5,2024-05-21,140.00,11980,INDIA,700.00,131,2,747.97,4,5,5,455,Champions


In [26]:
final_data.to_csv("consumer360_rfm_customer_segment.csv",index = False)